### Collection of things together now

In [2]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Access AWS credentials
access_key = os.getenv("ACCESS_KEY")
secret_key = os.getenv("SECRET_KEY")

In [3]:
import xarray as xr
import dask.array as da
from lightgbm import LGBMRegressor
from dask.distributed import LocalCluster, Client, performance_report
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import psutil
import os
from dask.diagnostics import ProgressBar
import warnings
from openeo.local import LocalConnection

warnings.filterwarnings(
    "ignore",
    module="sklearn.utils.validation"
)

if __name__ == '__main__':
    # Initialize Dask cluster and client inside the main block
    cluster = LocalCluster(
        n_workers=14,
        threads_per_worker=1,
        worker_dashboard_address=False,
        diagnostics_port=None
    )
    client = Client(cluster)
    
    # Initialize the local connection
    local_conn = LocalConnection("./")

    # Define the STAC collection URL
    stac_item = "https://stac.intertwin.fedcloud.eu/collections/ERA5_T2M_SSRD_TP"

    # Specify the spatial extent (bounding box)
    spatial_extent = {
        "west": 11.0,
        "east": 12.0,
        "south": 46.0,
        "north": 47.0
    }

    # Specify the temporal extent
    temporal_extent = ["2018-01-01", "2020-12-31"]

    # Load the data cube with specified parameters
    era5_single = local_conn.load_stac(
        url=stac_item,
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
    )
    
    stac_item = "https://stac.intertwin.fedcloud.eu/collections/ERA5_PRESSURE"

    era5_pressure = local_conn.load_stac(
        url=stac_item,
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
        bands=["t_850"]
    )
    
    stac_item = "https://stac.intertwin.fedcloud.eu/collections/EMO1_TA24_PR_RG_PET_DAILY"

    emo1 = local_conn.load_stac(
        url=stac_item,
        bands=["ta24"],
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
    )

    stac_item = "https://stac.intertwin.fedcloud.eu/collections/EMO1_DEM"

    dem = local_conn.load_stac(
        url=stac_item,
        spatial_extent=spatial_extent,
        bands=["dem"]
    )

    era5_cube = era5_single.merge_cubes(era5_pressure)
    #resample = dem.resample_spatial(resolution=0.666, method="bilinear", projection="EPSG:4326")
    remap = era5_cube.resample_cube_spatial(dem, method="bilinear")
    dem_expanded = dem.resample_cube_temporal(remap)
    print("REACHED HERE!")
    cube = remap.merge_cubes(dem_expanded).execute()
    cube = cube.to_dataset(dim="bands")
    cube# to go to raster_to_stac as UUID_X.zarr in the local

/home/sdhinakaran/micromamba/envs/zarr_downScaleML/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 34351 instead
  warnings.warn(


FileNotFoundError: No such file or directory: 's3://rucio/interTwin_EURAC/ERA5_T2M_SSRD_TP.zarr'

In [ ]:
import xarray as xr
import dask.array as da
from lightgbm import LGBMRegressor
from dask.distributed import LocalCluster, Client, performance_report
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import psutil
import os
from dask.diagnostics import ProgressBar
import warnings
from openeo.local import LocalConnection
import yaml

warnings.filterwarnings(
    "ignore",
    module="sklearn.utils.validation"
)

def load_config(config_path='config.yaml'):
    """Load configuration from YAML file."""
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    return config

def ensure_output_dir(output_dir):
    """Ensure output directory exists."""
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    return output_dir

if __name__ == '__main__':
    # Initialize Dask cluster and client inside the main block
    cluster = LocalCluster(
        n_workers=14,
        threads_per_worker=1,
        worker_dashboard_address=False,
        diagnostics_port=None
    )
    client = Client(cluster)

    config = load_config()

    # Get spatial and temporal extents from config
    spatial_extent = config.get('spatial_extent', {
        "west": 6.0,
        "east": 12.0,
        "south": 37.0,
        "north": 47.0
    })
    
    temporal_extent = config.get('temporal_extent', ["2005-01-01", "2020-12-31"])

    target_variable = config.get('target_variable', "ta24")

    output_dir = ensure_output_dir(config.get('output_directory', '/data'))
    
    # Initialize the local connection
    local_conn = LocalConnection("./")

    # Define the STAC collection URL
    stac_item = "https://stac.intertwin.fedcloud.eu/collections/ERA5_T2M_SSRD_TP"

    # Load the data cube with specified parameters
    era5_single = local_conn.load_stac(
        url=stac_item,
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
    )
    
    stac_item = "https://stac.intertwin.fedcloud.eu/collections/ERA5_PRESSURE"

    era5_pressure = local_conn.load_stac(
        url=stac_item,
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
        bands=["t_850"]
    )
    
    stac_item = "https://stac.intertwin.fedcloud.eu/collections/EMO1_TA24_PR_RG_PET_DAILY"

    emo1 = local_conn.load_stac(
        url=stac_item,
        bands=["ta24"],
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
    )

    stac_item = "https://stac.intertwin.fedcloud.eu/collections/EMO1_DEM"

    dem = local_conn.load_stac(
        url=stac_item,
        spatial_extent=spatial_extent,
        bands=["dem"]
    )

    era5_cube = era5_single.merge_cubes(era5_pressure)
    #resample = dem.resample_spatial(resolution=0.666, method="bilinear", projection="EPSG:4326")
    remap = era5_cube.resample_cube_spatial(dem, method="bilinear")
    dem_expanded = dem.resample_cube_temporal(remap)
    print("REACHED HERE!")
    cube = remap.merge_cubes(dem_expanded).execute()
    cube = cube.to_dataset(dim="bands")
    cube# to go to raster_to_stac as UUID_X.zarr in the local

In [ ]:
emo1_renamed = rename_labels(data=emo1, dimension = "bands", target = ["t2m","tp","ssrd"], source = ["ta24","pr","rg"])
y = emo1_resampled.to_dataset(dim="bands")
y = y[""] # target variable
y # To become a local stac item UUID


If incase SEAS5 is mentioned! be it a list of files or SEAS5 in general - Loop the same shit you do for ERA5

In [4]:
import xarray as xr

ds = xr.open_zarr("/mnt/CEPH_PROJECTS/InterTwin/Climate_Downscaling/PAPER/v1/ERA5_BASE/ERA5_BASE_DAILY_2000_2020_T2M_SSRD_TP.zarr")
ds

<xarray.Dataset> Size: 143MB
Dimensions:  (lat: 37, lon: 49, time: 6575)
Coordinates:
  * lat      (lat) float64 296B 51.0 50.75 50.5 50.25 ... 42.75 42.5 42.25 42.0
  * lon      (lon) float64 392B 4.0 4.25 4.5 4.75 5.0 ... 15.25 15.5 15.75 16.0
  * time     (time) datetime64[ns] 53kB 2000-01-01 2000-01-02 ... 2017-12-31
Data variables:
    ssrd     (time, lat, lon) float32 48MB dask.array<chunksize=(500, 37, 49), meta=np.ndarray>
    t2m      (time, lat, lon) float32 48MB dask.array<chunksize=(500, 37, 49), meta=np.ndarray>
    tp       (time, lat, lon) float32 48MB dask.array<chunksize=(500, 37, 49), meta=np.ndarray>
Attributes:
    crs:      EPSG:4326

In [10]:
ds = ds.sel(time=slice("2000-01-01", "2000-01-31"))

In [14]:
import xarray as xr
from datetime import datetime, timezone
from raster2stac import Raster2STAC
import logging
import os
import numpy as np

EURAC_RESEARCH_PROVIDER = {
                            "name": "Eurac Research - Institute for Earth Observation",
                            "url": "http://www.eurac.edu",
                            "roles": [
                            "processor"
                            ],
                        }
UUID = "12345"

rs2stac = Raster2STAC(
    data = ds, # Using only two variables to speed up the test
    write_collection_assets=True,
    collection_id = f"{UUID}_test_local_zarr", # The Collection id we want to set
    description = "EMO1 dataset for daily temperature, precipitation, solar radiation and potential evapo-transpiration calculated from daily temperature and solar radiation using Jensen Haise method for the years 2000 to 2022",
    license = "Apache-2.0",
    keywords = ["interTwin","EMO1","Zarr"],
    collection_url = "https://stac.intertwin.fedcloud.eu/collections/", # The URL where the collection will be 
    output_folder=f"/home/sdhinakaran/consoldiated_downScaleML/openEO-downScaleML/notebooks/{UUID}_",
    providers=[EURAC_RESEARCH_PROVIDER],
    sci_citation="Gomes, Goncalo; Thiemig, Vera; Skøien, Jon Olav; Ziese, Markus; Rauthe-Schöch, Armin; Rustemeier, Elke; Rehfeldt, Kira; Walawender, Jakub; Kolbe, Christine; Pichon, Damien; Schweim, Christoph; Salamon, Peter (2020): EMO: A high-resolution multi-variable gridded meteorological data set for Europe. European Commission, Joint Research Centre (JRC) [Dataset] doi: 10.2905/0BD84BE4-CEC8-4180-97A6-8B3ADAAC4D26 PID: http://data.europa.eu/89h/0bd84be4-cec8-4180-97a6-8b3adaac4d26",
    sci_doi="10.2905/0BD84BE4-CEC8-4180-97A6-8b3adaac4d26",
    links=[{"rel": "cite-as","href": "https://data.jrc.ec.europa.eu/dataset/0bd84be4-cec8-4180-97a6-8b3adaac4d26"}],
    s3_upload=False,
).generate_zarr_stac(item_id="test_local_zarr")

